# Model Development & Evaluation

> **DAY 2 — 모델 개발 및 평가**

DAY 1 EDA와 Feature Engineering 결과를 기반으로 배터리 Cycle Life 예측 모델을 개발하고 성능을 평가한다.

## Modeling Tasks

### Regression
- Target: `cycle_life`
- Metric: MAPE
- Paper Target: **9.1% MAPE**

### Classification
- Target: Cycle Life Class
- Metrics: F1-Score, Accuracy
- Paper Target: **95.1% Accuracy**

## Evaluation Strategy

- **Train:** Batch 1 (`2017-05-12`) — Cross-Validation
- **Valid:** Batch 1 (`2017-05-12`) — Cell-level Hold-out
- **Test:** Batch 2 (`2018-02-20`) — Independent Test Set
- **Additional Test:** Batch 3 — Optional

모델 선택과 전처리는 Batch 1 데이터만을 이용하며, Batch 2는 최종 일반화 성능 평가에만 사용한다.

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score
)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import (
    LinearRegression,
    LogisticRegression
)

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingRegressor,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    mean_absolute_percentage_error,
    accuracy_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Modeling 환경 설정 완료")

Modeling 환경 설정 완료


## 1. Data Preparation

Feature Engineering 단계에서 생성한 Cell 단위 Feature Dataset을 불러오고, 과제의 평가 기준에 따라 Batch 1과 Batch 2를 분리한다.

- **Batch 1 (`2017-05-12`)**: 모델 학습 및 검증
- **Batch 2 (`2018-02-20`)**: 최종 Test


In [41]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
FEATURE_PATH = PROJECT_ROOT / "results" / "feature_dataset.csv"

feature_df = pd.read_csv(FEATURE_PATH)

print("Feature Dataset shape:", feature_df.shape)
print("전체 Cell 수:", feature_df["cell_id"].nunique())

display(feature_df.head())

Feature Dataset shape: (141, 19)
전체 Cell 수: 141


,cell_id,batch,cycle_life,charging_policy,mean_QD,std_QD,QD_slope,deltaQ_mean,deltaQ_std,deltaQ_var,deltaQ_min,deltaQ_max,deltaQ_range,mean_IR,IR_slope,mean_Tavg,mean_Tmax,max_Tmax,mean_chargetime
0,2017-05-12_cell_0,2017-05-12,1190.0,3_6C-80PER_3_6C,0.280303,0.017804,-0.000074,-0.002873,0.003109,0.000010,-0.008460,0.000768,0.009228,0.016643,-6.107912e-07,31.603747,35.376748,35.994705,13.384231
1,2017-05-12_cell_1,2017-05-12,1179.0,3_6C-80PER_3_6C,0.280466,0.004920,0.000007,-0.004100,0.003112,0.000010,-0.011004,-0.000088,0.010916,0.016922,8.622394e-07,31.330314,34.160655,34.712265,13.374696
2,2017-05-12_cell_2,2017-05-12,1177.0,3_6C-80PER_3_6C,0.281207,0.004878,0.000032,-0.004487,0.004281,0.000018,-0.017216,0.000013,0.017229,0.016769,1.162834e-06,31.479584,34.544644,35.127342,13.378886
3,2017-05-12_cell_3,2017-05-12,1226.0,4C-80PER_4C,0.286239,0.005598,0.000010,-0.007456,0.006007,0.000036,-0.018961,0.000182,0.019143,0.016311,8.373194e-07,29.942199,30.662263,31.691414,12.053277
4,2017-05-12_cell_4,2017-05-12,1227.0,4C-80PER_4C,0.285034,0.006009,-0.000046,-0.005750,0.004744,0.000023,-0.013958,0.000046,0.014004,0.016669,2.768146e-07,31.448884,34.382516,35.651741,12.053314


In [42]:
from pathlib import Path

print("현재 위치:", Path.cwd())
print("feature_df 존재:", "feature_df" in globals())

if "feature_df" in globals():
    print("feature_df shape:", feature_df.shape)

현재 위치: /Users/seunghyunsuh/Desktop/SKALA 강의자료/12주차(09.28~10.02)/mini project/notebooks
feature_df 존재: True
feature_df shape: (141, 19)


In [43]:
print("cycle_life NaN 개수:", feature_df["cycle_life"].isna().sum())

print("\nBatch별 cycle_life NaN 개수:")
print(
    feature_df.groupby("batch")["cycle_life"]
    .apply(lambda x: x.isna().sum())
)

cycle_life NaN 개수: 12

Batch별 cycle_life NaN 개수:
batch
2017-05-12    0
2018-02-20    8
2018-04-03    2
2018-04-12    2
Name: cycle_life, dtype: int64


In [44]:
feature_df = pd.read_csv(FEATURE_PATH)

### 1.1 Train / Test Batch Split

과제의 평가 기준에 따라 Batch 1을 모델 개발용 데이터로, Batch 2를 최종 독립 Test 데이터로 분리한다.

- **Batch 1 (`2017-05-12`)**: Train / Validation
- **Batch 2 (`2018-02-20`)**: Final Test

Batch 2는 모델 선택 과정에 사용하지 않고 최종 일반화 성능 평가에만 사용한다.

In [45]:
batch1_df = feature_df[
    (feature_df["batch"] == "2017-05-12") &
    (feature_df["cycle_life"].notna())
].copy()

batch2_df = feature_df[
    (feature_df["batch"] == "2018-02-20") &
    (feature_df["cycle_life"].notna())
].copy()

print("Batch 1 shape:", batch1_df.shape)
print("Batch 2 shape:", batch2_df.shape)

print("\nBatch 1 Cell 수:", batch1_df["cell_id"].nunique())
print("Batch 2 Cell 수:", batch2_df["cell_id"].nunique())

Batch 1 shape: (46, 19)
Batch 2 shape: (39, 19)

Batch 1 Cell 수: 46
Batch 2 Cell 수: 39


## 2. Regression — Cycle Life Prediction

초기 100 Cycle에서 추출한 Feature를 이용하여 배터리의 최종 `cycle_life`를 예측한다.

### Candidate Models

- Linear Regression — Baseline
- Random Forest Regressor
- Gradient Boosting Regressor

### Evaluation

- Train: Batch 1 Cross-Validation
- Valid: Batch 1 Hold-out
- Test: Batch 2
- Metric: MAPE
- Paper Target: **9.1%**

In [46]:
feature_cols = [
    "mean_QD",
    "std_QD",
    "QD_slope",
    "deltaQ_mean",
    "deltaQ_std",
    "deltaQ_var",
    "deltaQ_min",
    "deltaQ_max",
    "deltaQ_range",
    "mean_IR",
    "IR_slope",
    "mean_Tavg",
    "mean_Tmax",
    "max_Tmax",
    "mean_chargetime"
]

X_batch1 = batch1_df[feature_cols].copy()
y_batch1 = batch1_df["cycle_life"].copy()

X_batch2 = batch2_df[feature_cols].copy()
y_batch2 = batch2_df["cycle_life"].copy()

print("Batch 1 X:", X_batch1.shape)
print("Batch 1 y:", y_batch1.shape)

print("\nBatch 2 X:", X_batch2.shape)
print("Batch 2 y:", y_batch2.shape)

print("\nBatch 1 Feature 결측치:")
display(X_batch1.isna().sum())

Batch 1 X: (46, 15)
Batch 1 y: (46,)

Batch 2 X: (39, 15)
Batch 2 y: (39,)

Batch 1 Feature 결측치:


mean_QD            0
std_QD             0
QD_slope           0
deltaQ_mean        0
deltaQ_std         0
deltaQ_var         0
deltaQ_min         0
deltaQ_max         0
deltaQ_range       0
mean_IR            0
IR_slope           0
mean_Tavg          0
mean_Tmax          0
max_Tmax           0
mean_chargetime    0
dtype: int64

### 2.1 Batch 1 Hold-out Split

Batch 1 내부에서 Cell 단위 Hold-out Validation Set을 구성한다.

- Train: Batch 1의 80%
- Validation: Batch 1의 20%
- Test: Batch 2 전체

모델 선택 및 전처리는 Batch 1 데이터만을 이용하며, Batch 2는 최종 평가 단계까지 사용하지 않는다.

In [47]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_batch1,
    y_batch1,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, y_train.shape)
print("Valid:", X_valid.shape, y_valid.shape)
print("Test :", X_batch2.shape, y_batch2.shape)

Train: (36, 15) (36,)
Valid: (10, 15) (10,)
Test : (39, 15) (39,)


### 2.2 Baseline Model — Linear Regression

가장 단순한 선형 회귀 모델을 Baseline으로 사용한다.

Feature의 스케일 차이를 고려하여 StandardScaler를 적용하고, 전처리 과정의 데이터 누수를 방지하기 위해 Pipeline으로 구성한다.

In [48]:
linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

train_pred = linear_model.predict(X_train)
valid_pred = linear_model.predict(X_valid)

train_mape = mean_absolute_percentage_error(
    y_train,
    train_pred
) * 100

valid_mape = mean_absolute_percentage_error(
    y_valid,
    valid_pred
) * 100

print("Linear Regression")
print(f"Train MAPE : {train_mape:.2f}%")
print(f"Valid MAPE : {valid_mape:.2f}%")
print(f"Train-Valid Gap : {valid_mape - train_mape:.2f}%p")

Linear Regression
Train MAPE : 5.31%
Valid MAPE : 5.62%
Train-Valid Gap : 0.31%p


### 2.3 Cross-Validation

Batch 1의 제한된 Cell 수에 따른 단일 Hold-out 결과의 변동성을 보완하기 위해 5-Fold Cross-Validation을 수행한다.

Cross-Validation은 Batch 1 데이터 내에서만 수행하며, 각 Fold의 MAPE와 평균 MAPE를 모델 비교 기준으로 사용한다.

In [49]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_scores = cross_val_score(
    linear_model,
    X_batch1,
    y_batch1,
    cv=cv,
    scoring="neg_mean_absolute_percentage_error"
)

cv_mape = -cv_scores * 100

print("Linear Regression — 5-Fold CV")
print("Fold MAPE:", np.round(cv_mape, 2))
print(f"Mean CV MAPE: {cv_mape.mean():.2f}%")
print(f"Std CV MAPE : {cv_mape.std():.2f}%")

Linear Regression — 5-Fold CV
Fold MAPE: [ 5.62  8.16  8.77  8.14 18.83]
Mean CV MAPE: 9.90%
Std CV MAPE : 4.59%


### 2.4 Random Forest Regressor

Feature와 Cycle Life 사이의 비선형 관계를 학습할 수 있는 Random Forest Regressor를 적용한다.

Linear Regression과 동일한 Train / Validation 및 5-Fold Cross-Validation 조건에서 성능을 비교한다.

In [50]:
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Hold-out 학습
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_valid_pred = rf_model.predict(X_valid)

rf_train_mape = mean_absolute_percentage_error(
    y_train,
    rf_train_pred
) * 100

rf_valid_mape = mean_absolute_percentage_error(
    y_valid,
    rf_valid_pred
) * 100

print("Random Forest")
print(f"Train MAPE : {rf_train_mape:.2f}%")
print(f"Valid MAPE : {rf_valid_mape:.2f}%")
print(f"Train-Valid Gap : {rf_valid_mape - rf_train_mape:.2f}%p")

Random Forest
Train MAPE : 3.38%
Valid MAPE : 5.97%
Train-Valid Gap : 2.60%p


In [51]:
rf_cv_scores = cross_val_score(
    rf_model,
    X_batch1,
    y_batch1,
    cv=cv,
    scoring="neg_mean_absolute_percentage_error"
)

rf_cv_mape = -rf_cv_scores * 100

print("Random Forest — 5-Fold CV")
print("Fold MAPE:", np.round(rf_cv_mape, 2))
print(f"Mean CV MAPE: {rf_cv_mape.mean():.2f}%")
print(f"Std CV MAPE : {rf_cv_mape.std():.2f}%")

Random Forest — 5-Fold CV
Fold MAPE: [ 6.06 10.95  9.07  9.61  4.87]
Mean CV MAPE: 8.11%
Std CV MAPE : 2.28%


### 2.5 Gradient Boosting Regressor

여러 개의 약한 예측 모델을 순차적으로 결합하여 이전 모델의 오차를 보완하는 Gradient Boosting Regressor를 적용한다.

Linear Regression 및 Random Forest와 동일한 Train / Validation 및 5-Fold Cross-Validation 조건에서 성능을 비교한다.

In [52]:
gb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE
    ))
])

gb_model.fit(X_train, y_train)

gb_train_pred = gb_model.predict(X_train)
gb_valid_pred = gb_model.predict(X_valid)

gb_train_mape = mean_absolute_percentage_error(
    y_train,
    gb_train_pred
) * 100

gb_valid_mape = mean_absolute_percentage_error(
    y_valid,
    gb_valid_pred
) * 100

print("Gradient Boosting")
print(f"Train MAPE : {gb_train_mape:.2f}%")
print(f"Valid MAPE : {gb_valid_mape:.2f}%")
print(f"Train-Valid Gap : {gb_valid_mape - gb_train_mape:.2f}%p")

Gradient Boosting
Train MAPE : 2.09%
Valid MAPE : 4.76%
Train-Valid Gap : 2.67%p


In [53]:
gb_cv_scores = cross_val_score(
    gb_model,
    X_batch1,
    y_batch1,
    cv=cv,
    scoring="neg_mean_absolute_percentage_error"
)

gb_cv_mape = -gb_cv_scores * 100

print("Gradient Boosting — 5-Fold CV")
print("Fold MAPE:", np.round(gb_cv_mape, 2))
print(f"Mean CV MAPE: {gb_cv_mape.mean():.2f}%")
print(f"Std CV MAPE : {gb_cv_mape.std():.2f}%")

Gradient Boosting — 5-Fold CV
Fold MAPE: [ 4.76  9.22 10.01  6.83  4.63]
Mean CV MAPE: 7.09%
Std CV MAPE : 2.22%


### 2.6 Regression Model Comparison

Batch 1의 5-Fold Cross-Validation과 Hold-out Validation 결과를 기준으로 세 회귀 모델의 성능을 비교한다.

모델 선택에는 독립 Test Set인 Batch 2를 사용하지 않는다.

In [54]:
regression_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "CV_MAPE": [
        cv_mape.mean(),
        rf_cv_mape.mean(),
        gb_cv_mape.mean()
    ],
    "CV_Std": [
        cv_mape.std(),
        rf_cv_mape.std(),
        gb_cv_mape.std()
    ],
    "Valid_MAPE": [
        valid_mape,
        rf_valid_mape,
        gb_valid_mape
    ]
})

regression_results = regression_results.sort_values(
    "CV_MAPE"
).reset_index(drop=True)

display(regression_results.round(2))

,Model,CV_MAPE,CV_Std,Valid_MAPE
0,Gradient Boosting,7.09,2.22,4.76
1,Random Forest,8.11,2.28,5.97
2,Linear Regression,9.90,4.59,5.62


#### Model Selection

Batch 1에서 세 회귀 모델을 비교한 결과, Gradient Boosting이 가장 낮은 평균 CV MAPE와 Hold-out Validation MAPE를 보였다.

따라서 Batch 2에 대한 최종 독립 Test 모델로 Gradient Boosting을 선택한다.

### 2.7 Final Test — Batch 2

Batch 1의 모델 비교 결과를 바탕으로 Gradient Boosting을 최종 회귀 모델로 선택한다.

최종 모델은 Batch 1 전체 데이터로 다시 학습한 후, 모델 선택 과정에서 사용하지 않은 Batch 2를 대상으로 독립 Test를 수행한다.

In [55]:
final_reg_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE
    ))
])

# Batch 1 전체로 최종 학습
final_reg_model.fit(X_batch1, y_batch1)

# Batch 2 최종 평가
test_pred = final_reg_model.predict(X_batch2)

test_mape = mean_absolute_percentage_error(
    y_batch2,
    test_pred
) * 100

cv_mean = gb_cv_mape.mean()
valid_mape_final = gb_valid_mape

print("=== Final Regression Performance ===")
print(f"Train (Batch 1 CV) : {cv_mean:.2f}%")
print(f"Valid (Hold-out)    : {valid_mape_final:.2f}%")
print(f"Test  (Batch 2)     : {test_mape:.2f}%")
print()
print(f"Train-Valid Gap     : {valid_mape_final - cv_mean:.2f}%p")
print(f"Valid-Test Gap      : {test_mape - valid_mape_final:.2f}%p")
print(f"Target-Test Gap     : {test_mape - 9.1:.2f}%p")
print("Paper Target        : 9.10%")

=== Final Regression Performance ===
Train (Batch 1 CV) : 7.09%
Valid (Hold-out)    : 4.76%
Test  (Batch 2)     : 33.26%

Train-Valid Gap     : -2.33%p
Valid-Test Gap      : 28.49%p
Target-Test Gap     : 24.16%p
Paper Target        : 9.10%


In [56]:
print("=== Cycle Life 분포 비교 ===")

print("\nBatch 1")
print(y_batch1.describe())

print("\nBatch 2")
print(y_batch2.describe())

print("\n=== 예측값 분포 ===")
print(pd.Series(test_pred).describe())

comparison = pd.DataFrame({
    "cell_id": batch2_df["cell_id"].values,
    "actual": y_batch2.values,
    "predicted": test_pred
})

comparison["APE_%"] = (
    np.abs(comparison["actual"] - comparison["predicted"])
    / comparison["actual"]
    * 100
)

display(
    comparison.sort_values(
        "APE_%",
        ascending=False
    ).head(15)
)

=== Cycle Life 분포 비교 ===

Batch 1
count      46.000000
mean      844.717391
std       184.629198
min       534.000000
25%       703.250000
50%       858.500000
75%       914.250000
max      1227.000000
Name: cycle_life, dtype: float64

Batch 2
count      39.000000
mean      565.743590
std       222.201629
min       392.000000
25%       439.500000
50%       472.000000
75%       508.500000
max      1186.000000
Name: cycle_life, dtype: float64

=== 예측값 분포 ===
count      39.000000
mean      705.445893
std       152.603952
min       562.104258
25%       606.101504
50%       636.583484
75%       725.187263
max      1080.669492
dtype: float64


,cell_id,actual,predicted,APE_%
18,2018-02-20_cell_18,449.0,728.457006,62.239868
33,2018-02-20_cell_41,442.0,703.975185,59.270404
19,2018-02-20_cell_19,392.0,623.044104,58.939822
21,2018-02-20_cell_21,408.0,624.826640,53.143784
2,2018-02-20_cell_2,424.0,647.828700,52.789788
6,2018-02-20_cell_6,393.0,596.509866,51.783681
15,2018-02-20_cell_15,396.0,595.895194,50.478584
28,2018-02-20_cell_30,412.0,611.284718,48.370077
27,2018-02-20_cell_29,452.0,660.478072,46.123467
38,2018-02-20_cell_46,503.0,721.917520,43.522370


In [57]:
# Classification threshold 결정을 위한 Cycle Life 분포 확인

print("=== Batch 1 Cycle Life ===")
print(y_batch1.describe())

print("\nMedian:", y_batch1.median())

print("\n=== 후보 Threshold별 클래스 수 ===")

for threshold in [700, 750, 800, 850, 900]:
    low = (y_batch1 < threshold).sum()
    high = (y_batch1 >= threshold).sum()

    print(
        f"{threshold} cycles → "
        f"Low: {low}, High: {high}"
    )

=== Batch 1 Cycle Life ===
count      46.000000
mean      844.717391
std       184.629198
min       534.000000
25%       703.250000
50%       858.500000
75%       914.250000
max      1227.000000
Name: cycle_life, dtype: float64

Median: 858.5

=== 후보 Threshold별 클래스 수 ===
700 cycles → Low: 10, High: 36
750 cycles → Low: 17, High: 29
800 cycles → Low: 20, High: 26
850 cycles → Low: 21, High: 25
900 cycles → Low: 32, High: 14


## 3. Classification — Cycle Life Class Prediction

Regression과 함께 배터리 수명을 범주형으로 예측하는 Classification 모델을 평가한다.

Batch 1의 Cycle Life 분포와 클래스 균형을 고려하여 **850 cycles**를 기준으로 두 클래스를 정의하였다.

- **Low Life (0)**: Cycle Life < 850
- **High Life (1)**: Cycle Life ≥ 850

### Candidate Models
- Logistic Regression
- Random Forest Classifier
- Gradient Boosting Classifier

### Evaluation
- Train: Batch 1 Cross-Validation
- Valid: Batch 1 Hold-out
- Test: Batch 2
- Metrics: Accuracy, F1-Score

In [58]:
# Classification Target
CLASS_THRESHOLD = 850

y_cls_batch1 = (y_batch1 >= CLASS_THRESHOLD).astype(int)
y_cls_batch2 = (y_batch2 >= CLASS_THRESHOLD).astype(int)

# Batch 1 Hold-out
X_cls_train, X_cls_valid, y_cls_train, y_cls_valid = train_test_split(
    X_batch1,
    y_cls_batch1,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_cls_batch1
)

# Models
classification_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=1000
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=2,
            random_state=RANDOM_STATE
        ))
    ])
}

# Stratified CV
from sklearn.model_selection import StratifiedKFold, cross_validate

cls_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

classification_results = []

for name, model in classification_models.items():

    cv_scores = cross_validate(
        model,
        X_batch1,
        y_cls_batch1,
        cv=cls_cv,
        scoring={
            "accuracy": "accuracy",
            "f1": "f1"
        }
    )

    model.fit(X_cls_train, y_cls_train)
    valid_pred = model.predict(X_cls_valid)

    classification_results.append({
        "Model": name,
        "CV_Accuracy": cv_scores["test_accuracy"].mean() * 100,
        "CV_F1": cv_scores["test_f1"].mean(),
        "Valid_Accuracy": accuracy_score(
            y_cls_valid, valid_pred
        ) * 100,
        "Valid_F1": f1_score(
            y_cls_valid, valid_pred
        )
    })

classification_results = pd.DataFrame(
    classification_results
).sort_values(
    "CV_Accuracy",
    ascending=False
).reset_index(drop=True)

display(classification_results.round(3))

,Model,CV_Accuracy,CV_F1,Valid_Accuracy,Valid_F1
0,Gradient Boosting,89.111,0.895,90.0,0.909
1,Random Forest,84.667,0.864,90.0,0.909
2,Logistic Regression,82.667,0.837,80.0,0.750


In [59]:
# ==========================================
# Final Classification Model
# Gradient Boosting → Batch 2 Test
# ==========================================

final_cls_model = classification_models["Gradient Boosting"]

# Batch 1 전체 데이터로 최종 학습
final_cls_model.fit(X_batch1, y_cls_batch1)

# Batch 2 최종 평가
test_cls_pred = final_cls_model.predict(X_batch2)

test_accuracy = accuracy_score(
    y_cls_batch2, test_cls_pred
) * 100

test_f1 = f1_score(
    y_cls_batch2, test_cls_pred
)

# 기존 CV / Valid 결과
gb_cls_result = classification_results[
    classification_results["Model"] == "Gradient Boosting"
].iloc[0]

train_cv_accuracy = gb_cls_result["CV_Accuracy"]
train_cv_f1 = gb_cls_result["CV_F1"]
valid_accuracy = gb_cls_result["Valid_Accuracy"]
valid_f1 = gb_cls_result["Valid_F1"]

# Gap
train_valid_gap = valid_accuracy - train_cv_accuracy
valid_test_gap = test_accuracy - valid_accuracy

# Paper Target
paper_accuracy_target = 95.1
target_test_gap = test_accuracy - paper_accuracy_target

print("=== Final Classification Performance ===")
print(f"Train (Batch 1 CV) Accuracy : {train_cv_accuracy:.2f}%")
print(f"Train (Batch 1 CV) F1       : {train_cv_f1:.3f}")
print(f"Valid (Hold-out) Accuracy    : {valid_accuracy:.2f}%")
print(f"Valid (Hold-out) F1          : {valid_f1:.3f}")
print(f"Test  (Batch 2) Accuracy     : {test_accuracy:.2f}%")
print(f"Test  (Batch 2) F1           : {test_f1:.3f}")

print()
print(f"Train-Valid Gap : {train_valid_gap:+.2f}%p")
print(f"Valid-Test Gap  : {valid_test_gap:+.2f}%p")
print(f"Target-Test Gap : {target_test_gap:+.2f}%p")
print(f"Paper Target    : {paper_accuracy_target:.2f}%")

=== Final Classification Performance ===
Train (Batch 1 CV) Accuracy : 89.11%
Train (Batch 1 CV) F1       : 0.895
Valid (Hold-out) Accuracy    : 90.00%
Valid (Hold-out) F1          : 0.909
Test  (Batch 2) Accuracy     : 92.31%
Test  (Batch 2) F1           : 0.667

Train-Valid Gap : +0.89%p
Valid-Test Gap  : +2.31%p
Target-Test Gap : -2.79%p
Paper Target    : 95.10%


## 4. Final Model Performance Summary

Regression과 Classification의 최종 성능을 정리하고 결과 파일로 저장한다.

In [60]:
# ==========================================
# Save Final Model Performance
# ==========================================

from pathlib import Path

performance_df = pd.DataFrame([
    {
        "Task": "Regression",
        "Model": "Gradient Boosting",
        "Metric": "MAPE",
        "Train_CV": 7.09,
        "Valid": 4.76,
        "Test": 33.26,
        "Paper_Target": 9.10,
        "Train_Valid_Gap": -2.33,
        "Valid_Test_Gap": 28.49,
        "Target_Test_Gap": 24.16
    },
    {
        "Task": "Classification",
        "Model": "Gradient Boosting",
        "Metric": "Accuracy",
        "Train_CV": 89.11,
        "Valid": 90.00,
        "Test": 92.31,
        "Paper_Target": 95.10,
        "Train_Valid_Gap": 0.89,
        "Valid_Test_Gap": 2.31,
        "Target_Test_Gap": -2.79
    }
])

RESULTS_DIR = Path.cwd().parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)

performance_path = RESULTS_DIR / "model_performance.csv"
performance_df.to_csv(performance_path, index=False)

display(performance_df)

print("\n저장 완료:", performance_path)
print("파일 존재:", performance_path.exists())

,Task,Model,Metric,Train_CV,Valid,Test,Paper_Target,Train_Valid_Gap,Valid_Test_Gap,Target_Test_Gap
0,Regression,Gradient Boosting,MAPE,7.09,4.76,33.26,9.1,-2.33,28.49,24.16
1,Classification,Gradient Boosting,Accuracy,89.11,90.00,92.31,95.1,0.89,2.31,-2.79



저장 완료: /Users/seunghyunsuh/Desktop/SKALA 강의자료/12주차(09.28~10.02)/mini project/results/model_performance.csv
파일 존재: True


## 5. Final Results & Interpretation

### 5-1. Regression 결과

초기 100 Cycle에서 추출한 Feature를 이용하여 Cycle Life를 예측한 결과,  
Gradient Boosting이 최종 Regression 모델로 선정되었다.

- **Train (Batch 1 CV) MAPE:** 7.09%
- **Validation (Batch 1 Hold-out) MAPE:** 4.76%
- **Test (Batch 2) MAPE:** 33.26%
- **Paper Target:** 9.10%

Batch 1 내부에서는 Paper Target인 9.10%보다 낮은 MAPE를 기록했으나,
독립적인 Batch 2 Test에서는 MAPE가 33.26%로 증가하였다.

특히 Validation-Test Gap이 **28.49%p**로 크게 나타나,
동일 Batch 내부에서는 비교적 정확한 Cycle Life 예측이 가능하지만
학습하지 않은 Batch로 데이터 분포가 변화할 경우 Regression 모델의
일반화 성능이 크게 저하되는 것을 확인하였다.

따라서 향후 Batch 간 차이를 설명할 수 있는 Charging Protocol 및 C-rate 등의
추가 Feature와 Batch 일반화를 고려한 모델 개선이 필요하다.


### 5-2. Classification 결과

Batch 1의 Cycle Life 분포와 클래스 균형을 고려하여
**850 cycles**를 Classification 기준으로 설정하였다.

- **Low Life (0):** Cycle Life < 850
- **High Life (1):** Cycle Life ≥ 850

세 가지 모델을 비교한 결과 Gradient Boosting이 가장 높은
Cross-Validation 성능을 보여 최종 Classification 모델로 선정되었다.

- **Train (Batch 1 CV) Accuracy:** 89.11%
- **Train (Batch 1 CV) F1:** 0.895
- **Validation Accuracy:** 90.00%
- **Validation F1:** 0.909
- **Test (Batch 2) Accuracy:** 92.31%
- **Test (Batch 2) F1:** 0.667
- **Paper Target Accuracy:** 95.10%

Batch 2 Test Accuracy는 92.31%로 Paper Target보다 2.79%p 낮았지만,
Regression과 비교하면 Batch 변화에도 Accuracy가 상대적으로 안정적으로 유지되었다.

다만 Test F1-score는 0.667로 Validation의 0.909보다 감소하였다.
따라서 Accuracy만으로 모델의 일반화 성능을 판단하기보다는
Batch 2의 클래스 분포 및 클래스별 예측 성능을 함께 고려할 필요가 있다.


### 5-3. 최종 결론

초기 100 Cycle 데이터만을 이용한 배터리 수명 예측 실험에서
Regression과 Classification 모두 Gradient Boosting이 최종 모델로 선정되었다.

Regression은 Batch 1 내부에서는 높은 예측 성능을 보였으나,
Batch 2에서 MAPE가 크게 증가하여 Batch 간 일반화에 한계가 나타났다.

반면 Classification은 Batch 2에서도 92.31%의 Accuracy를 기록하여
정확한 Cycle Life 수치를 예측하는 것보다 수명 범주를 구분하는 접근이
본 실험에서는 Batch 변화에 상대적으로 안정적인 결과를 보였다.

향후에는 Charging Protocol, C-rate 등 Batch 간 운용 조건의 차이를
Feature로 추가하고, 클래스별 Precision/Recall 및 Batch별 데이터 분포를
추가 분석하여 모델의 일반화 성능을 개선할 수 있다.